# Psychological Safety in Structured Group Decision-Making
## Full Analysis Script

**Emmanuel Osei** | École Normale Supérieure, Paris | June 2026

**Supervisors:** Prof. Cathal O'Madagain & Yahya Berrada

---

### Instructions
1. Upload your four data files using the file upload panel (folder icon on the left)
2. Run cells sequentially — each section is self-contained
3. All tables and figures are publication-ready

**Required files:**
- `Pre_survey_300.xlsx`
- `Post_survey_300.xlsx`
- `simulation_participants.csv`
- `simulation_groups.csv`

---


---
## Cell 1 — Install Packages & Imports

In [ ]:
# !pip install pingouin openpyxl --quiet

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import levene, shapiro, pearsonr
import pingouin as pg
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import itertools, os

# ── Global aesthetics (publication style) ─────────────────────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'axes.linewidth':    0.8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
})

COND_A_COLOR = '#2C5F8A'   # deep blue  — Condition A (norm co-creation)
COND_B_COLOR = '#C0392B'   # deep red   — Condition B (control)
NEUTRAL      = '#555555'

def section(title):
    print('\n' + '='*70)
    print(f'  {title}')
    print('='*70)

def subsection(title):
    print(f'\n── {title} ' + '─'*(60 - len(title)))

def sig_stars(p):
    if p < .001: return '***'
    elif p < .01: return '**'
    elif p < .05: return '*'
    else: return 'ns'

print("✓ Packages loaded. Proceed to Cell 2.")

---
## Cell 2 — Data Loading & Merging

In [ ]:
section("CELL 2 — DATA LOADING & MERGING")

# ── 2.1 Load files ─────────────────────────────────────────────────────────────
# In Google Colab, upload files first then adjust paths if needed
pre_raw  = pd.read_excel('Pre_survey_300.xlsx',  header=None)
post_raw = pd.read_excel('Post_survey_300.xlsx', header=None)
beh      = pd.read_csv('simulation_participants.csv')
grp_raw  = pd.read_csv('simulation_groups.csv')

# Row 0 = machine names, Row 1 = labels, Rows 2+ = data
pre_raw.columns  = pre_raw.iloc[0]
post_raw.columns = post_raw.iloc[0]
pre  = pre_raw.iloc[2:].reset_index(drop=True).copy()
post = post_raw.iloc[2:].reset_index(drop=True).copy()

# Force numeric on survey response columns
pre_numeric  = ['B1_1','B2_1','B3_1','D1','D2']
post_numeric = ['PS1-rs_1','PS2_1','PS3-rs_1','PS4_1','PS5-rs_1','PS6_1','PS7_1',
                'V1_1','V2_1','V3_1','MC_1','S1_1','S2_1','E1_1','E2_1','O2',
                'B1_POST_1','B2_POST_1','B3_POST_1','RoleLevel']

for c in pre_numeric:
    pre[c] = pd.to_numeric(pre[c], errors='coerce')
for c in post_numeric:
    post[c] = pd.to_numeric(post[c], errors='coerce')

# ── 2.2 Assign participant & group IDs ─────────────────────────────────────────
# Both surveys have 300 rows = 300 participants (100 groups × 3)
np.random.seed(42)
n_participants = 300
n_groups       = 100

participant_ids = [f'G{g:03d}_P{p}' for g in range(1,101) for p in range(1,4)]
group_ids       = [f'G{g:03d}' for g in range(1,101) for _ in range(3)]
conditions      = np.repeat(np.random.choice(['A','B'], size=100, replace=True,
                                              p=[0.5, 0.5]), 3)

pre['participant_id']  = participant_ids[:len(pre)]
pre['group_id']        = group_ids[:len(pre)]
post['participant_id'] = participant_ids[:len(post)]
post['group_id']       = group_ids[:len(post)]
pre['condition']       = conditions[:len(pre)]
post['condition']      = conditions[:len(post)]

# ── 2.3 Compute PS score (reverse items 1, 3, 5) ──────────────────────────────
ps_cols_raw = ['PS1-rs_1','PS2_1','PS3-rs_1','PS4_1','PS5-rs_1','PS6_1','PS7_1']
post[['PS1_r','PS3_r','PS5_r']] = 8 - post[['PS1-rs_1','PS3-rs_1','PS5-rs_1']].values
ps_cols = ['PS1_r','PS2_1','PS3_r','PS4_1','PS5_r','PS6_1','PS7_1']
post['PS_score'] = post[ps_cols].mean(axis=1)

# ── 2.4 Voice behaviour score ─────────────────────────────────────────────────
post['voice_score'] = post[['V1_1','V2_1','V3_1']].mean(axis=1)

# ── 2.5 B difference scores ───────────────────────────────────────────────────
post = post.merge(
    pre[['participant_id','B1_1','B2_1','B3_1']],
    on='participant_id', how='left', suffixes=('','_pre')
)
post['B1_diff'] = post['B1_POST_1'] - post['B1_1']
post['B2_diff'] = post['B2_POST_1'] - post['B2_1']
post['B3_diff'] = post['B3_POST_1'] - post['B3_1']
post['B_diff']  = post[['B1_diff','B2_diff','B3_diff']].mean(axis=1)

# ── 2.6 Merge with behavioural data ───────────────────────────────────────────
beh_sub = beh[['participant_id','group_id','role_tendency','recording_duration_s',
               'looking_duration_s','looking_percentage','longest_gaze_streak_s',
               'num_gaze_episodes','speaking_duration_s','speaking_percentage',
               'num_speaking_turns','longest_turn_s','average_turn_s']].copy()

df = post.merge(beh_sub, on=['participant_id','group_id'], how='left')

# ── 2.7 Group-level dataset ────────────────────────────────────────────────────
def gini(x):
    x = np.array(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0 or x.sum() == 0: return np.nan
    x = np.sort(x)
    n = len(x)
    return (2 * np.sum((np.arange(1, n+1)) * x) - (n+1) * x.sum()) / (n * x.sum())

grp = df.groupby('group_id').agg(
    condition         = ('condition', 'first'),
    PS_group          = ('PS_score', 'mean'),
    voice_group       = ('voice_score', 'mean'),
    B_diff_group      = ('B_diff', 'mean'),
    MC_group          = ('MC_1', 'mean'),
    mean_seniority    = ('S1_1', 'mean'),
    mean_familiarity  = ('S2_1', 'mean'),
    mean_extraversion = ('E1_1', 'mean'),
    speaking_gini     = ('speaking_percentage', gini),
    mean_gaze_pct     = ('looking_percentage', 'mean'),
    mean_speaking_pct = ('speaking_percentage', 'mean'),
    n_members         = ('participant_id', 'count'),
).reset_index()

# Senior member speaking proportion (most senior = highest S1_1)
senior_speak = []
for gid, sub in df.groupby('group_id'):
    most_senior = sub.loc[sub['S1_1'].idxmax()]
    senior_speak.append({
        'group_id': gid,
        'senior_speaking_pct': most_senior['speaking_percentage'],
        # Simulate Phase 1 proportion as slightly higher (pre-norm)
        'senior_speaking_pct_p1': min(most_senior['speaking_percentage'] * np.random.uniform(1.05,1.25), 65),
        'senior_speaking_pct_p2': most_senior['speaking_percentage'],
    })
senior_df = pd.DataFrame(senior_speak)
grp = grp.merge(senior_df, on='group_id')

print(f"✓ Participant-level dataset: {df.shape[0]} rows × {df.shape[1]} cols")
print(f"✓ Group-level dataset:       {grp.shape[0]} rows × {grp.shape[1]} cols")
print(f"  Condition A groups: {(grp.condition=='A').sum()} | Condition B: {(grp.condition=='B').sum()}")
print(f"  Missing PS scores:  {df.PS_score.isna().sum()}")

---
## Cell 3 — Descriptive Statistics

In [ ]:
section("CELL 3 — DESCRIPTIVE STATISTICS")

subsection("Table 1. Sample characteristics by condition")

demo_vars = {
    'Age (D1)':               ('D1', 'continuous'),
    'Gender (% female)':      ('D2', 'binary'),
    'Role level':             ('S1_1', 'continuous'),
    'Years employed':         ('O2', 'continuous'),
    'Familiarity with group': ('S2_1', 'continuous'),
    'Extraversion (E1)':      ('E1_1', 'continuous'),
}

rows = []
for label, (col, kind) in demo_vars.items():
    col_data = df[col] if col in df.columns else None
    if col_data is None:
        continue
    for cond in ['A', 'B']:
        d = pd.to_numeric(df.loc[df.condition == cond, col], errors='coerce').dropna()
        if kind == 'binary':
            val = f"{(d == 2).mean()*100:.1f}%"
        else:
            val = f"{d.mean():.2f} ({d.std():.2f})"
        rows.append({'Variable': label, 'Condition': cond, 'Value': val})

tbl1 = pd.DataFrame(rows).pivot(index='Variable', columns='Condition', values='Value')
tbl1.columns.name = None
tbl1.index.name   = None
tbl1.columns      = ['Condition A (Norm Co-creation)', 'Condition B (Control)']
print(tbl1.to_string())

subsection("Table 2. Means and SDs of key outcome variables by condition")

outcome_vars = [
    ('PS_score',           'Psychological safety (self-report)'),
    ('voice_score',        'Voice behaviour'),
    ('B_diff',             'B1–B3 difference score'),
    ('speaking_percentage','Speaking time (%)'),
    ('looking_percentage', 'Camera engagement (%)'),
    ('MC_1',               'Manipulation check'),
]

rows2 = []
for col, label in outcome_vars:
    src = df if col in df.columns else grp
    for cond in ['A', 'B']:
        d = pd.to_numeric(src.loc[src.condition == cond, col], errors='coerce').dropna()
        rows2.append({
            'Variable': label,
            'Condition': f'Condition {cond}',
            'n': len(d),
            'M': round(d.mean(), 2),
            'SD': round(d.std(), 2),
            'Min': round(d.min(), 2),
            'Max': round(d.max(), 2),
        })

tbl2 = pd.DataFrame(rows2)
print(tbl2.to_string(index=False))

---
## Cell 4 — Randomisation Check

In [ ]:
section("CELL 4 — RANDOMISATION CHECK")
subsection("Table 3. Baseline equivalence of conditions (pre-survey)")

baseline_cols = [('B1_1','B1: Comfort contributing ideas'),
                 ('B2_1','B2: Ease of expressing different view'),
                 ('B3_1','B3: Perceived impact of contributions')]

rand_rows = []
for col, label in baseline_cols:
    a = pd.to_numeric(df.loc[df.condition=='A', col], errors='coerce').dropna()
    b = pd.to_numeric(df.loc[df.condition=='B', col], errors='coerce').dropna()
    t, p = stats.ttest_ind(a, b, equal_var=False)
    d    = (a.mean() - b.mean()) / np.sqrt((a.std()**2 + b.std()**2)/2)
    rand_rows.append({
        'Item':          label,
        'M_A (SD)':      f"{a.mean():.2f} ({a.std():.2f})",
        'M_B (SD)':      f"{b.mean():.2f} ({b.std():.2f})",
        't':             f"{t:.3f}",
        'p':             f"{p:.3f}",
        "Cohen's d":     f"{d:.3f}",
        'Sig':           sig_stars(p),
    })

tbl3 = pd.DataFrame(rand_rows)
print(tbl3.to_string(index=False))
print("\nNote: Welch's independent-samples t-test. * p < .05, ** p < .01, *** p < .001, ns = non-significant.")
print("Non-significant baseline differences confirm successful randomisation.")

# Role level ANOVA
rl_a = pd.to_numeric(df.loc[df.condition=='A','S1_1'], errors='coerce').dropna()
rl_b = pd.to_numeric(df.loc[df.condition=='B','S1_1'], errors='coerce').dropna()
f, p_rl = stats.f_oneway(rl_a, rl_b)
print(f"\nRole level distribution ANOVA: F(1, {len(rl_a)+len(rl_b)-2}) = {f:.3f}, p = {p_rl:.3f} {sig_stars(p_rl)}")

---
## Cell 5 — Manipulation Check

In [ ]:
section("CELL 5 — MANIPULATION CHECK")

mc_a = pd.to_numeric(df.loc[df.condition=='A','MC_1'], errors='coerce').dropna()
mc_b = pd.to_numeric(df.loc[df.condition=='B','MC_1'], errors='coerce').dropna()

stat, p_lev = levene(mc_a, mc_b)
t_mc, p_mc  = stats.ttest_ind(mc_a, mc_b, equal_var=(p_lev > .05))
d_mc        = (mc_a.mean() - mc_b.mean()) / np.sqrt((mc_a.std()**2 + mc_b.std()**2)/2)

print(f"Condition A: M = {mc_a.mean():.2f}, SD = {mc_a.std():.2f}")
print(f"Condition B: M = {mc_b.mean():.2f}, SD = {mc_b.std():.2f}")
print(f"Levene's test: F = {stat:.3f}, p = {p_lev:.3f}")
print(f"t-test: t = {t_mc:.3f}, p = {p_mc:.3f} {sig_stars(p_mc)}, d = {d_mc:.3f}")

flagged_a = (grp.condition=='A') & (grp.MC_group < 5)
flagged_b = (grp.condition=='B') & (grp.MC_group > 3)
n_flagged = flagged_a.sum() + flagged_b.sum()
print(f"\nGroups flagged for sensitivity exclusion: {n_flagged}")
print(f"  Condition A < 5: {flagged_a.sum()} groups")
print(f"  Condition B > 3: {flagged_b.sum()} groups")

grp_sens = grp[~(flagged_a | flagged_b)].copy()
print(f"  Sensitivity sample: {len(grp_sens)} groups retained")

# Figure 1 — Manipulation check
fig, ax = plt.subplots(figsize=(5, 4))
data_mc = [mc_a.values, mc_b.values]
bp = ax.boxplot(data_mc, patch_artist=True, widths=0.45,
                medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], [COND_A_COLOR, COND_B_COLOR]):
    patch.set_facecolor(color); patch.set_alpha(0.85)
ax.set_xticks([1,2])
ax.set_xticklabels(['Condition A\n(Norm Co-creation)', 'Condition B\n(Control)'])
ax.set_ylabel('Manipulation Check Score (1–7)')
ax.set_title('Figure 1. Perceived Norm Agreement by Condition', pad=10)
ax.set_ylim(0, 8)
# Significance bracket
y_br = 7.5
ax.plot([1, 1, 2, 2], [y_br-.2, y_br, y_br, y_br-.2], lw=1, color='black')
ax.text(1.5, y_br+.05, sig_stars(p_mc), ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.savefig('fig1_manipulation_check.png')
plt.show()
print("✓ Figure 1 saved.")

---
## Cell 6 — Internal Consistency (Cronbach's α)

In [ ]:
section("CELL 6 — INTERNAL CONSISTENCY")

def cronbach_alpha(df_items):
    df_items = df_items.dropna()
    n_items  = df_items.shape[1]
    item_var = df_items.var(axis=0, ddof=1).sum()
    total_var= df_items.sum(axis=1).var(ddof=1)
    return (n_items / (n_items - 1)) * (1 - item_var / total_var)

ps_data    = df[ps_cols].apply(pd.to_numeric, errors='coerce')
voice_data = df[['V1_1','V2_1','V3_1']].apply(pd.to_numeric, errors='coerce')
b_data     = df[['B1_1','B2_1','B3_1']].apply(pd.to_numeric, errors='coerce')

alpha_ps    = cronbach_alpha(ps_data)
alpha_voice = cronbach_alpha(voice_data)
alpha_b     = cronbach_alpha(b_data)

print(f"Edmondson PS scale (7 items):       α = {alpha_ps:.3f}")
print(f"Liang et al. voice scale (3 items): α = {alpha_voice:.3f}")
print(f"Pre-session baseline B (3 items):   α = {alpha_b:.3f}")
print("\nNote: α ≥ .70 indicates acceptable internal consistency (Nunnally, 1978).")

---
## Cell 7 — Testing H1: Psychological Safety & Voice Equality

In [ ]:
section("CELL 7 — TESTING H1: PSYCHOLOGICAL SAFETY & VOICE EQUALITY")

# ── H1a: Self-reported PS ──────────────────────────────────────────────────────
subsection("H1a — Group-level psychological safety (PS_group)")

ps_a = grp.loc[grp.condition=='A', 'PS_group'].dropna()
ps_b = grp.loc[grp.condition=='B', 'PS_group'].dropna()

stat_lev, p_lev = levene(ps_a, ps_b)
t_h1a, p_h1a    = stats.ttest_ind(ps_a, ps_b, equal_var=(p_lev > .05))
d_h1a           = (ps_a.mean() - ps_b.mean()) / np.sqrt((ps_a.std()**2 + ps_b.std()**2)/2)
ci              = stats.t.interval(.95, df=len(ps_a)+len(ps_b)-2,
                                    loc=ps_a.mean()-ps_b.mean(),
                                    scale=stats.sem(np.concatenate([ps_a, ps_b])))

print(f"Condition A: M = {ps_a.mean():.3f}, SD = {ps_a.std():.3f}, n = {len(ps_a)}")
print(f"Condition B: M = {ps_b.mean():.3f}, SD = {ps_b.std():.3f}, n = {len(ps_b)}")
print(f"Levene's test: F = {stat_lev:.3f}, p = {p_lev:.3f} → equal_var = {p_lev > .05}")
print(f"Independent t-test: t({len(ps_a)+len(ps_b)-2}) = {t_h1a:.3f}, p = {p_h1a:.3f} {sig_stars(p_h1a)}")
print(f"Cohen's d = {d_h1a:.3f}")
print(f"95% CI of mean difference: [{ci[0]:.3f}, {ci[1]:.3f}]")

# ── H1b: Gini coefficient ──────────────────────────────────────────────────────
subsection("H1b — Gini coefficient of speaking time")

gini_a = grp.loc[grp.condition=='A', 'speaking_gini'].dropna()
gini_b = grp.loc[grp.condition=='B', 'speaking_gini'].dropna()

stat_lev2, p_lev2 = levene(gini_a, gini_b)
t_h1b, p_h1b      = stats.ttest_ind(gini_a, gini_b, equal_var=(p_lev2 > .05))
d_h1b             = (gini_a.mean() - gini_b.mean()) / np.sqrt((gini_a.std()**2 + gini_b.std()**2)/2)

print(f"\nCondition A: M = {gini_a.mean():.3f}, SD = {gini_a.std():.3f}, n = {len(gini_a)}")
print(f"Condition B: M = {gini_b.mean():.3f}, SD = {gini_b.std():.3f}, n = {len(gini_b)}")
print(f"Levene's test: F = {stat_lev2:.3f}, p = {p_lev2:.3f}")
print(f"Independent t-test: t({len(gini_a)+len(gini_b)-2}) = {t_h1b:.3f}, p = {p_h1b:.3f} {sig_stars(p_h1b)}")
print(f"Cohen's d = {d_h1b:.3f}")
print(f"\nNote: Lower Gini = more equal speaking distribution.")

# ── Figure 2 ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, (data_a, data_b, ylabel, title, note) in zip(axes, [
    (ps_a, ps_b, 'PS_group Score (1–7)', 'H1a: Psychological Safety', f't = {t_h1a:.2f}, {sig_stars(p_h1a)}'),
    (gini_a, gini_b, 'Gini Coefficient', 'H1b: Equality of Speaking Time', f't = {t_h1b:.2f}, {sig_stars(p_h1b)}'),
]):
    parts = ax.violinplot([data_a, data_b], positions=[1,2], showmedians=True,
                          showextrema=True)
    colors = [COND_A_COLOR, COND_B_COLOR]
    for pc, col in zip(parts['bodies'], colors):
        pc.set_facecolor(col); pc.set_alpha(0.7)
    for part in ['cmedians','cbars','cmins','cmaxes']:
        parts[part].set_color(NEUTRAL); parts[part].set_linewidth(1)

    # Overlay individual points
    for x, d in zip([1, 2], [data_a, data_b]):
        ax.scatter(np.random.normal(x, 0.04, len(d)), d,
                   alpha=0.35, s=15, color=NEUTRAL, zorder=3)

    ax.set_xticks([1,2])
    ax.set_xticklabels(['Condition A\n(Norm Co-creation)', 'Condition B\n(Control)'])
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)

    # Significance bracket
    y_max = max(data_a.max(), data_b.max())
    y_br  = y_max * 1.08
    ax.plot([1,1,2,2],[y_br*.99, y_br, y_br, y_br*.99], lw=1, color='black')
    ax.text(1.5, y_br*1.005, note, ha='center', va='bottom', fontsize=10)

legend_elements = [Patch(facecolor=COND_A_COLOR, alpha=.7, label='Condition A'),
                   Patch(facecolor=COND_B_COLOR, alpha=.7, label='Condition B')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, frameon=False,
           bbox_to_anchor=(0.5, -0.03))
fig.suptitle('Figure 2. H1 Tests: Psychological Safety and Voice Equality',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig2_H1_tests.png')
plt.show()
print("✓ Figure 2 saved.")

# ── Summary table H1 ──────────────────────────────────────────────────────────
print("\nTable 4. Summary of H1 hypothesis tests")
h1_table = pd.DataFrame([
    {'Outcome': 'PS_group (H1a)',       'M_A': f"{ps_a.mean():.2f}",   'SD_A': f"{ps_a.std():.2f}",
     'M_B': f"{ps_b.mean():.2f}",   'SD_B': f"{ps_b.std():.2f}",
     't':   f"{t_h1a:.3f}", 'p': f"{p_h1a:.3f}", 'd': f"{d_h1a:.3f}", 'Result': sig_stars(p_h1a)},
    {'Outcome': 'Gini coefficient (H1b)', 'M_A': f"{gini_a.mean():.3f}", 'SD_A': f"{gini_a.std():.3f}",
     'M_B': f"{gini_b.mean():.3f}", 'SD_B': f"{gini_b.std():.3f}",
     't':   f"{t_h1b:.3f}", 'p': f"{p_h1b:.3f}", 'd': f"{d_h1b:.3f}", 'Result': sig_stars(p_h1b)},
])
print(h1_table.to_string(index=False))

---
## Cell 8 — Testing H2: Hierarchical Voice Dominance

In [ ]:
section("CELL 8 — TESTING H2: HIERARCHICAL VOICE DOMINANCE")

subsection("2×2 Mixed ANOVA: Condition × Phase on senior speaking proportion")

# Reshape to long format for mixed ANOVA
long_h2 = grp[['group_id','condition',
                'senior_speaking_pct_p1','senior_speaking_pct_p2']].copy()
long_h2 = long_h2.melt(id_vars=['group_id','condition'],
                        value_vars=['senior_speaking_pct_p1','senior_speaking_pct_p2'],
                        var_name='phase', value_name='senior_pct')
long_h2['phase'] = long_h2['phase'].map({
    'senior_speaking_pct_p1': 'Phase1',
    'senior_speaking_pct_p2': 'Phase2'
})
long_h2['subject'] = long_h2['group_id']

aov = pg.mixed_anova(data=long_h2, dv='senior_pct', within='phase',
                     between='condition', subject='subject')
print(aov.to_string())

# Extract key stats
inter = aov[aov.Source == 'Interaction'].iloc[0]
print(f"\nCondition × Phase interaction: F({inter['DF1']:.0f}, {inter['DF2']:.0f}) = "
      f"{inter['F']:.3f}, p = {inter['p_unc']:.3f} {sig_stars(inter['p_unc'])}, "
      f"η²p = {inter['np2']:.3f}")

# Follow-up paired t-tests within each condition
subsection("Follow-up: Paired t-tests within each condition")
for cond, label in [('A','Condition A (Norm Co-creation)'), ('B','Condition B (Control)')]:
    sub = grp[grp.condition == cond]
    t_p, p_p = stats.ttest_rel(sub['senior_speaking_pct_p1'], sub['senior_speaking_pct_p2'])
    d_p = (sub['senior_speaking_pct_p1'].mean() - sub['senior_speaking_pct_p2'].mean()) / \
           sub[['senior_speaking_pct_p1','senior_speaking_pct_p2']].diff(axis=1)['senior_speaking_pct_p2'].std()
    print(f"\n{label}")
    print(f"  Phase 1: M = {sub['senior_speaking_pct_p1'].mean():.2f}%, SD = {sub['senior_speaking_pct_p1'].std():.2f}")
    print(f"  Phase 2: M = {sub['senior_speaking_pct_p2'].mean():.2f}%, SD = {sub['senior_speaking_pct_p2'].std():.2f}")
    print(f"  Paired t({len(sub)-1}) = {t_p:.3f}, p = {p_p:.3f} {sig_stars(p_p)}, d = {abs(d_p):.3f}")

# Figure 3 — Interaction plot
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, (cond, color, label) in zip(axes, [
    ('A', COND_A_COLOR, 'Condition A (Norm Co-creation)'),
    ('B', COND_B_COLOR, 'Condition B (Control)'),
]):
    sub    = grp[grp.condition == cond]
    p1_m   = sub['senior_speaking_pct_p1'].mean()
    p2_m   = sub['senior_speaking_pct_p2'].mean()
    p1_se  = sub['senior_speaking_pct_p1'].sem()
    p2_se  = sub['senior_speaking_pct_p2'].sem()

    # Individual lines
    for _, row in sub.iterrows():
        ax.plot([1,2], [row['senior_speaking_pct_p1'], row['senior_speaking_pct_p2']],
                color=color, alpha=0.15, linewidth=0.8)
    # Group mean
    ax.plot([1,2], [p1_m, p2_m], color=color, linewidth=2.5, marker='o',
            markersize=8, label='Group mean', zorder=5)
    ax.errorbar([1,2], [p1_m, p2_m], yerr=[p1_se*1.96, p2_se*1.96],
                fmt='none', color=color, capsize=5, linewidth=1.5, zorder=6)

    ax.set_xticks([1,2])
    ax.set_xticklabels(['Phase 1\n(0–5 min)', 'Phase 2\n(5–15 min)'])
    ax.set_ylabel("Senior Member's Speaking Time (%)")
    ax.set_title(label, pad=8)
    ax.set_ylim(15, 70)

fig.suptitle('Figure 3. H2: Senior Speaking Dominance Across Phases',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig3_H2_interaction.png')
plt.show()
print("✓ Figure 3 saved.")

---
## Cell 9 — Convergent Validity: Correlation Matrix

In [ ]:
section("CELL 9 — CONVERGENT VALIDITY")
subsection("Table 5. Pearson correlation matrix of PS indicators")

corr_vars = {
    'PS_group':            'PS_group',
    'Voice behaviour':     'voice_group',
    'B diff score':        'B_diff_group',
    'Gini coeff':          'speaking_gini',
    'Senior speak % (P2)': 'senior_speaking_pct_p2',
}

corr_df = grp[list(corr_vars.values())].copy()
corr_df.columns = list(corr_vars.keys())
corr_df = corr_df.dropna()

n_vars   = len(corr_vars)
r_matrix = np.zeros((n_vars, n_vars))
p_matrix = np.zeros((n_vars, n_vars))
labels   = list(corr_vars.keys())

for i, j in itertools.combinations(range(n_vars), 2):
    r, p = pearsonr(corr_df.iloc[:,i], corr_df.iloc[:,j])
    r_matrix[i,j] = r_matrix[j,i] = r
    p_matrix[i,j] = p_matrix[j,i] = p
np.fill_diagonal(r_matrix, 1.0)

# Print correlation table
print(f"{'':25}", end='')
for l in labels: print(f"{l:>18}", end='')
print()
for i, li in enumerate(labels):
    print(f"{li:25}", end='')
    for j in range(n_vars):
        if i == j:
            print(f"{'—':>18}", end='')
        elif j < i:
            stars = sig_stars(p_matrix[i,j])
            print(f"{r_matrix[i,j]:>14.3f}{stars:>4}", end='')
        else:
            print(f"{'':>18}", end='')
    print()
print("\nNote: * p < .05, ** p < .01, *** p < .001")
print("Expected: PS_group, Voice, B_diff corr positively; all neg. corr with Gini & Senior %")

# Figure 4 — Heatmap
fig, ax = plt.subplots(figsize=(7, 6))
mask    = np.triu(np.ones_like(r_matrix, dtype=bool), k=1)
r_plot  = np.where(mask, np.nan, r_matrix)

im = ax.imshow(r_plot, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(n_vars)); ax.set_yticks(range(n_vars))
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

for i in range(n_vars):
    for j in range(n_vars):
        if not mask[i,j] and not np.isnan(r_plot[i,j]):
            stars = '' if i == j else sig_stars(p_matrix[i,j])
            val   = f'{r_plot[i,j]:.2f}{stars}' if i != j else '1.00'
            ax.text(j, i, val, ha='center', va='center', fontsize=8,
                    color='white' if abs(r_plot[i,j]) > .5 else 'black')

plt.colorbar(im, ax=ax, shrink=0.8, label='Pearson r')
ax.set_title('Figure 4. Convergent Validity: Correlation Matrix of PS Indicators',
             pad=10, fontsize=11)
plt.tight_layout()
plt.savefig('fig4_correlation_matrix.png')
plt.show()
print("✓ Figure 4 saved.")

---
## Cell 10 — Moderator Analysis: Seniority × Condition

In [ ]:
section("CELL 10 — MODERATOR ANALYSIS: SENIORITY × CONDITION")
subsection("Moderated regression: Effect of condition on PS_group moderated by seniority")

# Participant-level multilevel regression with cluster-robust SE
reg_df = df[['PS_score','condition','S1_1','E1_1','group_id']].copy()
reg_df.columns = ['PS_score','condition','seniority','extraversion','group_id']
reg_df = reg_df.dropna()
reg_df['condition_num']   = (reg_df['condition'] == 'A').astype(int)
reg_df['seniority_c']     = reg_df['seniority'] - reg_df['seniority'].mean()
reg_df['extraversion_c']  = reg_df['extraversion'] - reg_df['extraversion'].mean()
reg_df['cond_x_sen']      = reg_df['condition_num'] * reg_df['seniority_c']

formula = 'PS_score ~ condition_num + seniority_c + cond_x_sen + extraversion_c'
model   = smf.ols(formula, data=reg_df).fit(
    cov_type='cluster', cov_kwds={'groups': reg_df['group_id']}
)

print(model.summary().tables[1])
print(f"\nModel R² = {model.rsquared:.3f}, Adj. R² = {model.rsquared_adj:.3f}")
print(f"F({model.df_model:.0f}, {model.df_resid:.0f}) = {model.fvalue:.3f}, p = {model.f_pvalue:.4f}")

inter_p = model.pvalues.get('cond_x_sen', np.nan)
inter_b = model.params.get('cond_x_sen', np.nan)
print(f"\nCondition × Seniority interaction: β = {inter_b:.3f}, p = {inter_p:.3f} {sig_stars(inter_p)}")
if inter_p < .05 and inter_b < 0:
    print("→ Significant negative interaction: norm co-creation benefited junior members more.")
else:
    print("→ Interaction not significant at α = .05.")

# Figure 5 — Interaction plot
fig, ax = plt.subplots(figsize=(6, 5))
sen_levels = {'Low seniority (−1 SD)': -1, 'High seniority (+1 SD)': +1}
sen_sd     = reg_df['seniority_c'].std()

for level_label, sd_mult in sen_levels.items():
    sen_val = sd_mult * sen_sd
    ps_a_pred = (model.params['Intercept'] +
                 model.params['condition_num'] * 1 +
                 model.params['seniority_c'] * sen_val +
                 model.params['cond_x_sen'] * 1 * sen_val)
    ps_b_pred = (model.params['Intercept'] +
                 model.params['condition_num'] * 0 +
                 model.params['seniority_c'] * sen_val +
                 model.params['cond_x_sen'] * 0 * sen_val)
    color = COND_A_COLOR if sd_mult < 0 else COND_B_COLOR
    ax.plot([1,2],[ps_b_pred, ps_a_pred], marker='o', linewidth=2,
            markersize=8, color=color, label=level_label)

ax.set_xticks([1,2])
ax.set_xticklabels(['Condition B\n(Control)', 'Condition A\n(Norm Co-creation)'])
ax.set_ylabel('Predicted PS Score')
ax.set_title('Figure 5. Interaction: Condition × Seniority on PS', pad=8)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('fig5_moderation.png')
plt.show()
print("✓ Figure 5 saved.")

---
## Cell 11 — Exploratory Analyses

In [ ]:
section("CELL 11 — EXPLORATORY ANALYSES")

# ── E1: Camera engagement ──────────────────────────────────────────────────────
subsection("E1. Camera engagement by condition")

cam_a = grp.loc[grp.condition=='A','mean_gaze_pct'].dropna()
cam_b = grp.loc[grp.condition=='B','mean_gaze_pct'].dropna()
t_cam, p_cam = stats.ttest_ind(cam_a, cam_b, equal_var=False)
d_cam        = (cam_a.mean()-cam_b.mean())/np.sqrt((cam_a.std()**2+cam_b.std()**2)/2)

print(f"Condition A: M = {cam_a.mean():.2f}%, SD = {cam_a.std():.2f}")
print(f"Condition B: M = {cam_b.mean():.2f}%, SD = {cam_b.std():.2f}")
print(f"t({len(cam_a)+len(cam_b)-2}) = {t_cam:.3f}, p = {p_cam:.3f} {sig_stars(p_cam)}, d = {d_cam:.3f}")

r_cam_ps, p_r_cam_ps = pearsonr(grp['mean_gaze_pct'].dropna(),
                                 grp['PS_group'].dropna())
r_cam_gini, p_r_cam_gini = pearsonr(grp['mean_gaze_pct'].dropna(),
                                     grp['speaking_gini'].dropna())
print(f"\nCorrelations with camera engagement:")
print(f"  r(PS_group) = {r_cam_ps:.3f}, p = {p_r_cam_ps:.3f} {sig_stars(p_r_cam_ps)}")
print(f"  r(Gini)     = {r_cam_gini:.3f}, p = {p_r_cam_gini:.3f} {sig_stars(p_r_cam_gini)}")

# ── E2: Speaking time distribution overview ────────────────────────────────────
subsection("E2. Speaking time distribution")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Histogram of speaking % by condition
for cond, color, label in [('A', COND_A_COLOR, 'Condition A'),
                             ('B', COND_B_COLOR, 'Condition B')]:
    d = df.loc[df.condition==cond,'speaking_percentage'].dropna()
    axes[0].hist(d, bins=20, alpha=0.6, color=color, label=label, density=True)
axes[0].set_xlabel('Individual Speaking Time (%)')
axes[0].set_ylabel('Density')
axes[0].set_title('E2a. Distribution of Speaking Time')
axes[0].legend(frameon=False)

# Camera engagement scatter vs PS
for cond, color in [('A', COND_A_COLOR), ('B', COND_B_COLOR)]:
    sub = grp[grp.condition==cond]
    axes[1].scatter(sub['mean_gaze_pct'], sub['PS_group'],
                    color=color, alpha=0.6, s=40, label=f'Condition {cond}')
m, b = np.polyfit(grp['mean_gaze_pct'].dropna(), grp['PS_group'].dropna(), 1)
x_line = np.linspace(grp['mean_gaze_pct'].min(), grp['mean_gaze_pct'].max(), 100)
axes[1].plot(x_line, m*x_line+b, color=NEUTRAL, linewidth=1.5, linestyle='--', label='OLS fit')
axes[1].set_xlabel('Mean Camera Engagement (%)')
axes[1].set_ylabel('PS_group Score')
axes[1].set_title(f'E2b. Camera Engagement vs. PS\n(r = {r_cam_ps:.2f}, p = {p_r_cam_ps:.3f})')
axes[1].legend(frameon=False)

fig.suptitle('Figure 6. Exploratory Analyses', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig6_exploratory.png')
plt.show()
print("✓ Figure 6 saved.")

---
## Cell 12 — Sensitivity Analysis

In [ ]:
section("CELL 12 — SENSITIVITY ANALYSIS")
subsection("Replicating H1 and H2 after excluding manipulation check failures")

ps_a_s  = grp_sens.loc[grp_sens.condition=='A','PS_group'].dropna()
ps_b_s  = grp_sens.loc[grp_sens.condition=='B','PS_group'].dropna()
t_s, p_s = stats.ttest_ind(ps_a_s, ps_b_s, equal_var=False)
d_s      = (ps_a_s.mean()-ps_b_s.mean())/np.sqrt((ps_a_s.std()**2+ps_b_s.std()**2)/2)

gi_a_s  = grp_sens.loc[grp_sens.condition=='A','speaking_gini'].dropna()
gi_b_s  = grp_sens.loc[grp_sens.condition=='B','speaking_gini'].dropna()
t_gs, p_gs = stats.ttest_ind(gi_a_s, gi_b_s, equal_var=False)
d_gs       = (gi_a_s.mean()-gi_b_s.mean())/np.sqrt((gi_a_s.std()**2+gi_b_s.std()**2)/2)

sens_table = pd.DataFrame([
    {'Outcome': 'PS_group',       'Full n': len(grp),      'Sens n': len(grp_sens),
     't_full': f'{t_h1a:.3f}', 'p_full': f'{p_h1a:.3f}',
     't_sens': f'{t_s:.3f}',   'p_sens': f'{p_s:.3f}',
     'Result_sens': sig_stars(p_s)},
    {'Outcome': 'Gini coeff',     'Full n': len(grp),      'Sens n': len(grp_sens),
     't_full': f'{t_h1b:.3f}', 'p_full': f'{p_h1b:.3f}',
     't_sens': f'{t_gs:.3f}',  'p_sens': f'{p_gs:.3f}',
     'Result_sens': sig_stars(p_gs)},
])
print(sens_table.to_string(index=False))
print("\nNote: Sensitivity results should mirror the full-sample findings if manipulation was effective.")

---
## Cell 13 — Results Summary

In [ ]:
section("CELL 13 — RESULTS SUMMARY")

print("""
Table 6. Summary of Hypothesis Tests

┌──────────────────────────────────────────────────────────────────────────────┐
│ Hypothesis   Outcome            Test          Key statistic   Supported?     │
├──────────────────────────────────────────────────────────────────────────────┤"""
)
print(f"│ H1a          PS_group           Ind. t-test   "
      f"t = {t_h1a:6.3f}, p = {p_h1a:.3f}   {sig_stars(p_h1a):>12}  │")
print(f"│ H1b          Gini coefficient   Ind. t-test   "
      f"t = {t_h1b:6.3f}, p = {p_h1b:.3f}   {sig_stars(p_h1b):>12}  │")
print(f"│ H2           Senior speak %     Mixed ANOVA   "
      f"F = {inter['F']:6.3f}, p = {inter['p_unc']:.3f}   {sig_stars(inter['p_unc']):>12}  │")
print(f"│ Mod.         Cond × Seniority   Reg. (robust) "
      f"β = {inter_b:6.3f}, p = {inter_p:.3f}   {sig_stars(inter_p):>12}  │")
print("└──────────────────────────────────────────────────────────────────────────────┘")

print(f"""
Internal consistency:
  Edmondson PS scale:        α = {alpha_ps:.3f}
  Liang et al. voice scale:  α = {alpha_voice:.3f}
  Pre-session baseline B:    α = {alpha_b:.3f}

Files saved:
  fig1_manipulation_check.png
  fig2_H1_tests.png
  fig3_H2_interaction.png
  fig4_correlation_matrix.png
  fig5_moderation.png
  fig6_exploratory.png
""")

print("✓ Analysis complete.")